In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from Secret_Key import gemini_key
import os
import pyodbc
import re

In [2]:
os.environ['GEMINI_API_KEY'] = gemini_key

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-robotics-er-1.5-preview",temperature=0.7)

In [4]:
prompt = f"""
You are an expert Microsoft SQL Server data analyst.

DATABASE DETAILS
Database name: atliq_tshirts
Database type: Microsoft SQL Server

SCHEMA (THIS IS THE COMPLETE SCHEMA — DO NOT INVENT ANY TABLES)

Table: t_shirts
Columns:
- t_shirt_id (int, primary key)
- brand (Van Huesen, Levi, Nike, Adidas)
- color (Red, Blue, Black, White)
- size (XS, S, M, L, XL)
- price (int)
- stock_quantity (int)

Table: discounts
Columns:
- discount_id (int, primary key)
- t_shirt_id (int, foreign key → t_shirts.t_shirt_id)
- pct_discount (decimal between 0 and 100)

IMPORTANT RULES (MUST FOLLOW ALL)
- Target database is SQL Server
- Use SQL Server syntax ONLY
- NEVER use backticks (`)
- NEVER use MySQL syntax
- NEVER use LIMIT (use TOP instead)
- Use only the tables and columns listed above
- DO NOT assume the existence of sales or orders tables
- There is NO sales history in this database
- “Sales” must be DERIVED using available columns
- DO NOT use Markdown
- DO NOT wrap output in ``` blocks
- Output ONLY raw SQL

DEFINITION OF "TOTAL SALES"
Total sales means:
price × stock_quantity

QUESTION
If we have to sell all the Levi’s T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?

Return ONLY the SQL Server query.

"""

In [24]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-V5VTL7B;"
    "DATABASE=atliq_tshirts;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()

for row in cursor.tables(tableType='TABLE'):
    print(f"Table Name: {row.table_name}, Schema: {row.table_schem}, Catalog: {row.table_cat}")

for row in cursor.columns(table='discounts'):
    print(row.column_name)

for row in cursor.columns(table='t_shirts'):
    print(row.column_name)

Table Name: discounts, Schema: dbo, Catalog: atliq_tshirts
Table Name: t_shirts, Schema: dbo, Catalog: atliq_tshirts
Table Name: trace_xe_action_map, Schema: sys, Catalog: atliq_tshirts
Table Name: trace_xe_event_map, Schema: sys, Catalog: atliq_tshirts
discount_id
t_shirt_id
pct_discount
t_shirt_id
brand
color
size
price
stock_quantity


In [6]:
raw_sql = llm.invoke(prompt).content
print("Raw SQL:\n", raw_sql)

Raw SQL:
 SELECT
    SUM(t.price * t.stock_quantity * (1 - ISNULL(d.pct_discount, 0) / 100.0))
FROM
    t_shirts AS t
LEFT JOIN
    discounts AS d ON t.t_shirt_id = d.t_shirt_id
WHERE
    t.brand = 'Levi';


In [7]:
def clean_sql(sql: str) -> str:
    sql = sql.strip()
    # Remove ```sql ... ``` or ``` ... ```
    sql = re.sub(r"^```(?:sql)?", "", sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r"```$", "", sql).strip()
    return sql

In [8]:
sql = clean_sql(raw_sql)
print("Clean SQL:\n", sql)

Clean SQL:
 SELECT
    SUM(t.price * t.stock_quantity * (1 - ISNULL(d.pct_discount, 0) / 100.0))
FROM
    t_shirts AS t
LEFT JOIN
    discounts AS d ON t.t_shirt_id = d.t_shirt_id
WHERE
    t.brand = 'Levi';


In [9]:
cursor = conn.cursor()
if "`" in sql:
    raise ValueError("Invalid SQL dialect: backticks detected")
cursor.execute(sql)
rows = cursor.fetchall()

for row in rows:
    print(row)

(Decimal('35238.6900000'),)


## Few-Shot Learning

In [10]:
few_shots = [
    {'Question' : "How many t-shirts do we have left for Nike in XS size and white color?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
     'SQLResult': "Result of the SQL query",
     'Answer' : 70},
    {'Question': "How much is the total price of the inventory for all S-size t-shirts?",
     'SQLQuery':"SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S'",
     'SQLResult': "Result of the SQL query",
     'Answer': 21013},
    {'Question': "If we have to sell all the Levi’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)?" ,
     'SQLQuery' : """SELECT sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
 """,
     'SQLResult': "Result of the SQL query",
     'Answer': 35238.690000} ,
     {'Question' : "If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?" ,
      'SQLQuery': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
      'SQLResult': "Result of the SQL query",
      'Answer' : 37656},
    {'Question': "How many white color Levi's shirt I have?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'",
     'SQLResult': "Result of the SQL query",
     'Answer' : 289
     }
]

### Creating Semantic Similarity Based example selector

* Create embedding on the few_shots
* Store the embeddings in Chroma DB
* Retrieve the the top most Semantically close example from the vector store

In [15]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

to_vectorize = [" ".join(map(str, example.values())) for example in few_shots]

In [13]:
to_vectorize

["How many t-shirts do we have left for Nike in XS size and white color? SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS' Result of the SQL query 70",
 "How much is the total price of the inventory for all S-size t-shirts? SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S' Result of the SQL query 21013",
 "If we have to sell all the Levi’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)? SELECT sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from\n(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'\ngroup by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id\n  Result of the SQL query 35238.69",
 "If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount? SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'

In [14]:
vectorstore = FAISS.from_texts(
    to_vectorize,
    embeddings,
    metadatas=few_shots
)
vectorstore

In [16]:
example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=2,  #k-> No. of examples
)

example_selector.select_examples({"Question": "How many Adidas T shirts I have left in my store?"})

[{'Question': 'How many t-shirts do we have left for Nike in XS size and white color?',
  'SQLQuery': "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
  'SQLResult': 'Result of the SQL query',
  'Answer': 70},
 {'Question': 'If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?',
  'SQLQuery': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
  'SQLResult': 'Result of the SQL query',
  'Answer': 37656}]

### Taking user query outside the prompt

In [32]:
prompt = f"""
You are an expert Microsoft SQL Server data analyst.

DATABASE DETAILS
Database name: atliq_tshirts
Database type: Microsoft SQL Server

SCHEMA (THIS IS THE COMPLETE SCHEMA — DO NOT INVENT ANY TABLES)

Table: t_shirts
Columns:
- t_shirt_id (int, primary key)
- brand (Van Huesen, Levi, Nike, Adidas)
- color (Red, Blue, Black, White)
- size (XS, S, M, L, XL)
- price (int)
- stock_quantity (int)

Table: discounts
Columns:
- discount_id (int, primary key)
- t_shirt_id (int, foreign key → t_shirts.t_shirt_id)
- pct_discount (decimal between 0 and 100)

IMPORTANT RULES (MUST FOLLOW ALL)
- Target database is SQL Server
- Use SQL Server syntax ONLY
- NEVER use backticks (`)
- NEVER use MySQL syntax
- NEVER use LIMIT (use TOP instead)
- Use only the tables and columns listed above
- DO NOT assume the existence of sales or orders tables
- There is NO sales history in this database
- “Sales” must be DERIVED using available columns
- DO NOT use Markdown
- DO NOT wrap output in ``` blocks
- Output ONLY raw SQL

DEFINITION OF "TOTAL SALES"
Total sales means:
price × stock_quantity

QUESTION
{input}

Return ONLY the SQL Server query.

"""

In [37]:
prompt_text = few_shot_prompt.format(
    input="How much is the price of the inventory for all small size t-shirts?",
    top_k=5
)

In [ ]:
raw_sql = llm.invoke(prompt_text).content
print("Raw SQL:\n", raw_sql)

Raw SQL:
 SELECT SUM(price * stock_quantity) AS total_sales
FROM t_shirts;


In [39]:
sql = clean_sql(raw_sql)
print("Clean SQL:\n", sql)

Clean SQL:
 SELECT SUM(price * stock_quantity) AS total_sales
FROM t_shirts;


In [40]:
cursor = conn.cursor()
if "`" in sql:
    raise ValueError("Invalid SQL dialect: backticks detected")
cursor.execute(sql)
rows = cursor.fetchall()

for row in rows:
    print(row)

(128856,)
